# Регистрация Inobitec и электродов

**Статус:** исполняемый валидатор контракта регистрации; сама регистрация
по имеющимся в репозитории данным не выполнена.

Цель этапа — связать принятый DICOM-манифест, экспортированные ручные
маски Inobitec, поверхность тела и координаты электродов в одной явно
заданной системе координат. Результат этого ноутбука подтверждает
прослеживаемость и формальную согласованность передачи данных, но не создаёт
маски, не подбирает преобразование и не доказывает анатомическую точность.


## Что уже реализовано и чего нет

Каноническая сегментация выполнена вручную в Inobitec, однако её файлы и
метаданные экспорта находятся вне Git. В репозитории нет проверенного
преобразования из DICOM LPS в систему координат STL и нет единого манифеста
связи с электродами.

MATLAB-ветка частично использует уже подготовленную геометрию:

- MATLAB_TRKG4_real_subjects/src/trkg4_subject_registry.m перечисляет
  STL тела и органов;
- MATLAB_TRKG4_real_subjects/src/trkg4_load_electrode_centres.m читает
  четыре центра электродов в миллиметрах;
- MATLAB_TRKG4_real_subjects/src/run_trkg4_inverse_inhale.m подбирает
  смещение и угол решётки на уже заданной поверхности.

Последний расчёт является модельной подгонкой позы, а не регистрацией
масок Inobitec. Его результат нельзя использовать как доказательство
происхождения исходной системы координат.


## Обязательный контракт

Для каждого испытуемого требуются:

1. принятый манифест 20.01 той же КТ-серии;
2. внешние файлы ручных масок с ролями, версией Inobitec, способом
   сегментации и SHA-256;
3. система координат экспорта, единицы и матрица 4×4 из DICOM LPS в
   систему экспорта;
4. источник преобразования и отдельно оценённая ошибка регистрации;
5. для каждого исследования — отдельный CSV четырёх электродов с
   колонками label, x_mm, y_mm, z_mm;
6. площадка, прибор, монтаж, источник исходной геометрии электродов и
   отдельные составляющие её неопределённости.

Эксперименты 2 и 3 описываются разными монтажами даже для одного
испытуемого. Одинаковая схема электродов не означает одинаковую позу,
прибор, сессию или калибровку.

Вход 20.02 содержит только исходную или референтную геометрию по прямому
измерению либо реконструкции протокола. Поза, подобранная по импедансным
данным, является результатом 20.10 и не возвращается сюда как независимый
вход.


## Проверки и границы результата

Валидатор проверяет наличие файлов, хеши, обязательные роли масок,
ортонормальность линейной части преобразования между миллиметровыми
системами координат, структуру CSV и точный порядок электродов:
I_plus, V_plus, V_minus, I_minus.

Значения неопределённости не объединяются одним словом «шум». Отдельно
задаются ошибка границы ручной сегментации, остаток регистрации,
погрешность ручной установки электродов, экспорта координат и проекции на
поверхность. До получения обоснованных чисел поля остаются пустыми, а
манифест не может получить статус accepted.

Автоматические проверки не оценивают визуальное совпадение поверхностей.
Принятие требует отдельного ручного просмотра наложения масок и электродов
на анатомию.


In [ ]:
import csv
import hashlib
import json
import math
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

ALGORITHM_VERSION = "registration_handoff_v1"
ELECTRODE_LABELS = ["I_plus", "V_plus", "V_minus", "I_minus"]
CONFIG_PATH = Path(os.environ["KALMYKOV_CT_CONFIG"]).expanduser().resolve()
CONFIG_BYTES = CONFIG_PATH.read_bytes()
CONFIG = json.loads(CONFIG_BYTES.decode("utf-8"))
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
SUBJECT_SPECS = CONFIG["subjects"]
TRANSFORM_TOLERANCE = float(
    CONFIG["registration_qc"]["orthonormal_tolerance"]
)

subject_ids = [item["subject_id"] for item in SUBJECT_SPECS]
if len(subject_ids) != len(set(subject_ids)):
    raise ValueError("subject_id должны быть уникальными")


In [ ]:
def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def finite_nonnegative(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) and number >= 0 else None


def determinant_3x3(matrix):
    return (
        matrix[0][0]
        * (matrix[1][1] * matrix[2][2] - matrix[1][2] * matrix[2][1])
        - matrix[0][1]
        * (matrix[1][0] * matrix[2][2] - matrix[1][2] * matrix[2][0])
        + matrix[0][2]
        * (matrix[1][0] * matrix[2][1] - matrix[1][1] * matrix[2][0])
    )


def validate_transform(matrix):
    errors = []
    if (
        not isinstance(matrix, list)
        or len(matrix) != 4
        or any(not isinstance(row, list) or len(row) != 4 for row in matrix)
    ):
        return None, [{"code": "transform_must_be_4x4"}]
    try:
        values = [[float(item) for item in row] for row in matrix]
    except (TypeError, ValueError):
        return None, [{"code": "transform_contains_non_numeric_values"}]
    if any(not math.isfinite(item) for row in values for item in row):
        return None, [{"code": "transform_contains_non_finite_values"}]
    if max(
        abs(values[3][index] - target)
        for index, target in enumerate([0.0, 0.0, 0.0, 1.0])
    ) > TRANSFORM_TOLERANCE:
        errors.append({"code": "invalid_homogeneous_last_row"})

    rotation = [row[:3] for row in values[:3]]
    gram = [
        [
            sum(rotation[k][i] * rotation[k][j] for k in range(3))
            for j in range(3)
        ]
        for i in range(3)
    ]
    orthonormal_error = max(
        abs(gram[i][j] - (1.0 if i == j else 0.0))
        for i in range(3)
        for j in range(3)
    )
    determinant = determinant_3x3(rotation)
    if orthonormal_error > TRANSFORM_TOLERANCE:
        errors.append(
            {
                "code": "transform_linear_part_not_orthonormal",
                "max_error": orthonormal_error,
            }
        )
    if abs(abs(determinant) - 1.0) > TRANSFORM_TOLERANCE:
        errors.append(
            {
                "code": "transform_determinant_not_unit",
                "determinant": determinant,
            }
        )
    return values, errors


def read_electrodes(path):
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        required = {"label", "x_mm", "y_mm", "z_mm"}
        if reader.fieldnames is None or set(reader.fieldnames) != required:
            raise ValueError(
                "CSV электродов должен содержать только "
                "label,x_mm,y_mm,z_mm"
            )
        rows = list(reader)
    if [row["label"] for row in rows] != ELECTRODE_LABELS:
        raise ValueError(
            "Порядок электродов должен быть "
            + ", ".join(ELECTRODE_LABELS)
        )
    coordinates = []
    for row in rows:
        xyz = [float(row[field]) for field in ("x_mm", "y_mm", "z_mm")]
        if any(not math.isfinite(item) for item in xyz):
            raise ValueError("Координаты электродов должны быть конечными")
        coordinates.append(xyz)
    if len(coordinates) != 4:
        raise ValueError("Требуются ровно четыре электрода")
    return coordinates


In [ ]:
def build_registration_manifest(subject):
    errors = []
    warnings = []
    subject_id = subject["subject_id"]
    registration = subject.get("registration")
    if not isinstance(registration, dict):
        registration = {}
        errors.append({"code": "missing_registration_section"})

    dicom_manifest_path = (
        DERIVED_ROOT / "ct" / "dicom_qc" / f"{subject_id}.json"
    )
    if not dicom_manifest_path.is_file():
        raise FileNotFoundError(
            f"Нет DICOM-манифеста 20.01 для {subject_id}"
        )
    dicom_manifest_bytes = dicom_manifest_path.read_bytes()
    dicom_manifest = json.loads(dicom_manifest_bytes.decode("utf-8"))
    if dicom_manifest.get("algorithm_version") != "dicom_qc_geometry_v1":
        errors.append({"code": "unsupported_dicom_manifest_version"})
    if dicom_manifest.get("qc", {}).get("status") != "accepted":
        errors.append({"code": "dicom_manifest_not_accepted"})
    if dicom_manifest.get("subject_id") != subject_id:
        errors.append({"code": "dicom_subject_mismatch"})

    frame_units = registration.get("frame_units")
    if frame_units != "mm":
        errors.append({"code": "export_frame_units_must_be_mm"})
    frame_name = str(registration.get("frame_name", "")).strip()
    if not frame_name:
        errors.append({"code": "missing_export_frame_name"})
    transform_source = str(
        registration.get("transform_source", "")
    ).strip()
    if not transform_source:
        errors.append({"code": "missing_transform_source"})
    transform, transform_errors = validate_transform(
        registration.get("dicom_lps_to_export_4x4")
    )
    errors.extend(transform_errors)

    segmentation_uncertainty = finite_nonnegative(
        registration.get("segmentation_boundary_uncertainty_mm")
    )
    if segmentation_uncertainty is None:
        errors.append(
            {"code": "missing_segmentation_boundary_uncertainty_mm"}
        )
    transform_residual = finite_nonnegative(
        registration.get("transform_residual_mm")
    )
    if transform_residual is None:
        errors.append({"code": "missing_transform_residual_mm"})

    required_roles = set(registration.get("required_mask_roles", []))
    if not required_roles:
        errors.append({"code": "missing_required_mask_role_contract"})
    mask_specs = registration.get("manual_masks", [])
    mask_roles = [str(item.get("role", "")).strip() for item in mask_specs]
    if len(mask_roles) != len(set(mask_roles)):
        errors.append({"code": "duplicate_mask_roles"})
    missing_roles = sorted(required_roles - set(mask_roles))
    if missing_roles:
        errors.append(
            {"code": "missing_required_mask_roles", "roles": missing_roles}
        )

    masks = []
    source_file_hashes = []
    for mask in mask_specs:
        role = str(mask.get("role", "")).strip()
        mask_format = str(mask.get("format", "")).strip().lower()
        if not role:
            errors.append({"code": "missing_mask_role"})
        if not mask_format:
            errors.append({"code": "missing_mask_format", "role": role})
        mask_path = Path(mask.get("path", "")).expanduser().resolve()
        if not mask_path.is_file():
            errors.append({"code": "missing_mask_file", "role": role})
            continue
        if mask.get("segmentation_method") != "manual":
            errors.append(
                {"code": "mask_not_declared_manual", "role": role}
            )
        if str(mask.get("source_application", "")).strip() != "Inobitec":
            errors.append(
                {"code": "mask_source_not_inobitec", "role": role}
            )
        if not str(mask.get("source_application_version", "")).strip():
            errors.append(
                {"code": "missing_inobitec_version", "role": role}
            )
        file_hash = sha256_file(mask_path)
        source_file_hashes.append(file_hash)
        masks.append(
            {
                "role": role,
                "format": mask_format,
                "segmentation_method": "manual",
                "source_application": "Inobitec",
                "source_application_version": str(
                    mask.get("source_application_version")
                ),
                "file_sha256": file_hash,
            }
        )

    montage_specs = registration.get("montages", [])
    montage_ids = [
        str(item.get("montage_id", "")).strip() for item in montage_specs
    ]
    if not montage_specs:
        errors.append({"code": "missing_montages"})
    if len(montage_ids) != len(set(montage_ids)):
        errors.append({"code": "duplicate_montage_ids"})

    montages = []
    required_uncertainties = {
        "manual_placement_mm",
        "coordinate_export_mm",
        "surface_projection_mm",
    }
    for montage in montage_specs:
        montage_id = str(montage.get("montage_id", "")).strip()
        if not montage_id:
            errors.append({"code": "missing_montage_id"})
        experiment_id = str(montage.get("experiment_id", "")).strip()
        instrument_id = str(montage.get("instrument_id", "")).strip()
        if experiment_id not in {"exp02", "exp03"}:
            errors.append(
                {
                    "code": "unsupported_experiment_id",
                    "montage_id": montage_id,
                }
            )
        if not instrument_id:
            errors.append(
                {
                    "code": "missing_instrument_id",
                    "montage_id": montage_id,
                }
            )
        reference_geometry_source_type = montage.get("reference_geometry_source_type")
        if reference_geometry_source_type not in {
            "manual_measurement",
            "protocol_reconstruction",
        }:
            errors.append(
                {
                    "code": "invalid_reference_geometry_source_type",
                    "montage_id": montage_id,
                }
            )
        if not str(montage.get("reference_geometry_source_reference", "")).strip():
            errors.append(
                {
                    "code": "missing_reference_geometry_source_reference",
                    "montage_id": montage_id,
                }
            )

        electrode_path = Path(
            montage.get("electrode_centres_csv", "")
        ).expanduser().resolve()
        coordinates = None
        electrode_hash = None
        if not electrode_path.is_file():
            errors.append(
                {
                    "code": "missing_electrode_csv",
                    "montage_id": montage_id,
                }
            )
        else:
            try:
                coordinates = read_electrodes(electrode_path)
                electrode_hash = sha256_file(electrode_path)
                source_file_hashes.append(electrode_hash)
            except (OSError, ValueError) as exc:
                errors.append(
                    {
                        "code": "invalid_electrode_csv",
                        "montage_id": montage_id,
                        "detail": str(exc),
                    }
                )

        uncertainties = montage.get("uncertainty", {})
        normalized_uncertainties = {}
        for field in sorted(required_uncertainties):
            value = finite_nonnegative(uncertainties.get(field))
            normalized_uncertainties[field] = value
            if value is None:
                errors.append(
                    {
                        "code": "missing_montage_uncertainty",
                        "montage_id": montage_id,
                        "field": field,
                    }
                )
        montages.append(
            {
                "montage_id": montage_id,
                "experiment_id": experiment_id,
                "instrument_id": instrument_id,
                "reference_geometry_source_type": reference_geometry_source_type,
                "reference_geometry_source_reference": montage.get(
                    "reference_geometry_source_reference"
                ),
                "electrode_order": ELECTRODE_LABELS,
                "electrode_centres_xyz_mm": coordinates,
                "electrode_file_sha256": electrode_hash,
                "uncertainty": normalized_uncertainties,
            }
        )

    config_sha256 = sha256_bytes(CONFIG_BYTES)
    source_parts = sorted(
        [
            sha256_bytes(dicom_manifest_bytes),
            config_sha256,
            *source_file_hashes,
        ]
    )
    source_bundle_sha256 = sha256_bytes(
        ("\n".join(source_parts) + "\n").encode("ascii")
    )
    return {
        "schema_version": 1,
        "algorithm_version": ALGORITHM_VERSION,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "subject_id": subject_id,
        "source": {
            "config_sha256": config_sha256,
            "dicom_manifest_sha256": sha256_bytes(dicom_manifest_bytes),
            "dicom_source_set_sha256": dicom_manifest.get(
                "source", {}
            ).get("source_set_sha256"),
            "source_bundle_sha256": source_bundle_sha256,
        },
        "registration": {
            "source_coordinate_system": "DICOM patient LPS",
            "export_frame_name": frame_name,
            "frame_units": frame_units,
            "dicom_lps_to_export_4x4": transform,
            "transform_source": transform_source,
            "segmentation_boundary_uncertainty_mm": (
                segmentation_uncertainty
            ),
            "transform_residual_mm": transform_residual,
            "manual_masks": masks,
            "montages": montages,
        },
        "runtime": {"python": platform.python_version()},
        "qc": {
            "status": "pending_manual_review",
            "automatic_errors": errors,
            "automatic_warnings": warnings,
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
            "history": [],
        },
    }


## Формирование кандидатного манифеста передачи данных

Для каждого испытуемого создаётся внешний файл
derived_root/ct/registration/<subject_id>.json. Он содержит только
обезличенные идентификаторы, хеши, преобразование, координаты электродов,
происхождение и раздельные неопределённости. Исходные пути в результат не
записываются.

Существующий манифест не перезаписывается. Изменение КТ-манифеста,
конфигурации, маски или CSV электродов требует архивирования прежнего
результата и повторного ручного контроля.


In [ ]:
MANIFESTS = {}
output_root = DERIVED_ROOT / "ct" / "registration"
output_root.mkdir(parents=True, exist_ok=True)

for subject in SUBJECT_SPECS:
    candidate = build_registration_manifest(subject)
    subject_id = subject["subject_id"]
    output_path = output_root / f"{subject_id}.json"
    if output_path.exists():
        existing = json.loads(output_path.read_text(encoding="utf-8"))
        same_bundle = (
            existing.get("source", {}).get("source_bundle_sha256")
            == candidate["source"]["source_bundle_sha256"]
        )
        same_algorithm = (
            existing.get("algorithm_version") == ALGORITHM_VERSION
        )
        if not same_bundle or not same_algorithm:
            raise RuntimeError(
                f"Манифест {subject_id} относится к другому входу или "
                "версии; сначала архивируйте его"
            )
        MANIFESTS[subject_id] = existing
        print(
            "Сохранён существующий манифест:",
            subject_id,
            existing["qc"]["status"],
        )
        continue

    output_path.write_text(
        json.dumps(candidate, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    MANIFESTS[subject_id] = candidate
    print(
        "Создан кандидатный манифест:",
        subject_id,
        "ошибок:",
        len(candidate["qc"]["automatic_errors"]),
    )


## Ручной контроль и принятие

Статус accepted требует отсутствия автоматических ошибок, имени
проверяющего, подтверждённого визуального наложения масок на исходную КТ,
подтверждённого происхождения преобразования и отдельного подтверждения
каждого монтажа. Решение не означает, что референтная геометрия совпадает с
фактической позой во время записи, что последующая модельная подгонка
единственна или что FEM уже валидирован.


In [ ]:
REVIEW_DECISIONS = {
    # "nik": {
    #     "status": "accepted",  # accepted или rejected
    #     "reviewer": "<reviewer>",
    #     "confirmed_visual_alignment": True,
    #     "confirmed_transform_provenance": True,
    #     "confirmed_montage_ids": [
    #         "exp02_mgtu_side",
    #         "exp03_rnch_side",
    #     ],
    #     "notes": "<основание решения>",
    # },
}

for subject_id, decision in REVIEW_DECISIONS.items():
    if subject_id not in MANIFESTS:
        raise KeyError(f"Неизвестный subject_id: {subject_id}")
    manifest = MANIFESTS[subject_id]
    if manifest["qc"]["status"] == "accepted":
        print("Уже принят, без перезаписи:", subject_id)
        continue

    status = decision.get("status")
    reviewer = str(decision.get("reviewer", "")).strip()
    if status not in {"accepted", "rejected"} or not reviewer:
        raise ValueError(
            "Нужны статус accepted/rejected и имя проверяющего"
        )
    if status == "accepted":
        if manifest["qc"]["automatic_errors"]:
            raise ValueError(
                f"{subject_id}: нельзя принять манифест с ошибками"
            )
        if decision.get("confirmed_visual_alignment") is not True:
            raise ValueError("Требуется подтвердить визуальное наложение")
        if decision.get("confirmed_transform_provenance") is not True:
            raise ValueError(
                "Требуется подтвердить происхождение преобразования"
            )
        expected_montages = {
            item["montage_id"]
            for item in manifest["registration"]["montages"]
        }
        confirmed_montages = set(
            decision.get("confirmed_montage_ids", [])
        )
        if confirmed_montages != expected_montages:
            raise ValueError(
                "Нужно отдельно подтвердить каждый montage_id"
            )

    previous_qc = manifest["qc"]
    history = list(previous_qc.get("history", []))
    history.append(
        {
            "status": previous_qc.get("status"),
            "reviewer": previous_qc.get("reviewer"),
            "reviewed_at": previous_qc.get("reviewed_at"),
            "notes": previous_qc.get("notes"),
        }
    )
    manifest["qc"] = {
        "status": status,
        "automatic_errors": previous_qc["automatic_errors"],
        "automatic_warnings": previous_qc["automatic_warnings"],
        "reviewer": reviewer,
        "reviewed_at": datetime.now(timezone.utc).isoformat(),
        "notes": decision.get("notes"),
        "history": history,
    }
    output_path = output_root / f"{subject_id}.json"
    output_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("Решение сохранено:", subject_id, status)


## Выход и следующие этапы

Канонический выход — принятый внешний манифест регистрации той же версии
и того же набора входов. Его используют:

- 20.03 — для связи полного ручного ROI лёгкого с геометрией КТ;
- 20.10 — как источник происхождения STL и начальной геометрии электродов;
- MATLAB_TRKG4_real_subjects — после отдельной проверки соответствия
  локальному реестру и конкретному запуску.

До появления заполненного и принятого манифеста регистрация считается
отсутствующей. Наличие STL, CSV или результата модельной подгонки по
отдельности не закрывает этот этап.
